In [1]:
import argparse
import os
import os.path as osp
import warnings
from copy import deepcopy

from mmengine import ConfigDict
from mmengine.config import Config, DictAction
from mmengine.runner import Runner

from mmdet.engine.hooks.utils import trigger_visualization_hook
from mmdet.evaluation import DumpDetResults
from mmdet.registry import RUNNERS
from mmdet.utils import setup_cache_size_limit_of_dynamo

In [2]:
import torch

In [3]:
config_f = '../../configs/pretrain/yolo_world_v2_audio32k_obj365_fineaudio_fast_align.py'
                        # yolo_world_v2_audio32k_obj365_fineaudio_fast_align

In [4]:
cfg = Config.fromfile(config_f)
cfg.work_dir = './test'

for d in cfg.train_dataloader['dataset']['datasets']:
    d['dataset']['data_root'] = '../../' + d['dataset']['data_root']
    # d['data_root'] = '../../' + d['data_root']
    d['class_text_path'] = '../../' + d['class_text_path']
    d['class_audio_path'] = '../../' + d['class_audio_path']
    
runner = Runner.from_cfg(cfg)

11/04 10:32:39 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
11/04 10:32:40 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.17 (default, Jul  5 2023, 21:04:15) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1132249475
    GPU 0,1,2,3,4,5,6,7: GeForce RTX 3090
    CUDA_HOME: /usr/local/cuda-11.2
    NVCC: Cuda compilation tools, release 11.2, V11.2.67
    GCC: gcc (GCC) 7.3.1 20180303 (Red Hat 7.3.1-5)
    PyTorch: 2.0.1+cu117
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Libra

In [5]:
cfg.train_dataloader['dataset']['datasets'][0]

{'type': 'MultiModalDataset',
 'dataset': {'type': 'YOLOv5Objects365V1Dataset',
  'data_root': '../../data/objects365v1/',
  'ann_file': 'annotations/objects365_train_131k.json',
  'data_prefix': {'img': 'train/'},
  'filter_cfg': {'filter_empty_gt': False, 'min_size': 32}},
 'class_text_path': '../../data/texts/obj365v1_class_texts.json',
 'class_audio_path': '../../data/audios/obj365v1_class_audio22.json',
 'pipeline': [{'type': 'LoadImageFromFile', 'backend_args': None},
  {'type': 'LoadAnnotations', 'with_bbox': True},
  {'type': 'MultiModalMosaic',
   'img_scale': (640, 640),
   'pad_val': 114.0,
   'pre_transform': [{'type': 'LoadImageFromFile', 'backend_args': None},
    {'type': 'LoadAnnotations', 'with_bbox': True}]},
  {'type': 'YOLOv5RandomAffine',
   'max_rotate_degree': 0.0,
   'max_shear_degree': 0.0,
   'scaling_ratio_range': (0.5, 1.5),
   'max_aspect_ratio': 100,
   'border': (-320, -320),
   'border_val': (114, 114, 114)},
  {'type': 'mmdet.Albu',
   'transforms': [{'

In [5]:
model = runner.model
# getattr(self, i)

# backbone.image_model

In [10]:
for param in model.backbone.audio_model.parameters():
    print(param.requires_grad, end=' ')
# model.backbone.image_model.parameters

False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False False Fals

In [9]:
model.backbone

MultiModalAudioYOLOBackbone(
  (image_model): YOLOv8CSPDarknet(
    (stem): ConvModule(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (activate): SiLU(inplace=True)
    )
    (stage1): Sequential(
      (0): ConvModule(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (activate): SiLU(inplace=True)
      )
      (1): CSPLayerWithTwoConv(
        (main_conv): ConvModule(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (activate): SiLU(inplace=True)
        )
        (final_conv): ConvModule(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=

In [11]:
ms = 'backbone.image_model'
m = model

for s in ms.split('.'):
    m = getattr(m, s)
    
# getattr(model, 'backbone.image_model')

In [12]:
m

YOLOv8CSPDarknet(
  (stem): ConvModule(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (activate): SiLU(inplace=True)
  )
  (stage1): Sequential(
    (0): ConvModule(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (activate): SiLU(inplace=True)
    )
    (1): CSPLayerWithTwoConv(
      (main_conv): ConvModule(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (activate): SiLU(inplace=True)
      )
      (final_conv): ConvModule(
        (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_ru

In [12]:
cfg.train_dataloader['dataset']['datasets']

[{'type': 'MultiModalDataset',
  'dataset': {'type': 'YOLOv5Objects365V1Dataset',
   'data_root': '../../data/objects365v1/',
   'ann_file': 'annotations/objects365_train_2k.json',
   'data_prefix': {'img': 'train/'},
   'filter_cfg': {'filter_empty_gt': False, 'min_size': 32}},
  'class_text_path': '../../data/texts/obj365v1_class_texts.json',
  'class_audio_path': '../../data/audios/obj365v1_class_audio.json',
  'pipeline': [{'type': 'LoadImageFromFile', 'backend_args': None},
   {'type': 'LoadAnnotations', 'with_bbox': True},
   {'type': 'MultiModalMosaic',
    'img_scale': (640, 640),
    'pad_val': 114.0,
    'pre_transform': [{'type': 'LoadImageFromFile', 'backend_args': None},
     {'type': 'LoadAnnotations', 'with_bbox': True}]},
   {'type': 'YOLOv5RandomAffine',
    'max_rotate_degree': 0.0,
    'max_shear_degree': 0.0,
    'scaling_ratio_range': (0.5, 1.5),
    'max_aspect_ratio': 100,
    'border': (-320, -320),
    'border_val': (114, 114, 114)},
   {'type': 'mmdet.Albu',
 

In [7]:
train_loader = runner.train_dataloader

loading annotations into memory...
Done (t=11.18s)
creating index...
index created!


In [ ]:
train_loader.dataset.datasets[0].pipeline.transforms[-3].class_audios

In [8]:
for b in train_loader:
    print(b)
    break

{'data_samples': {'bboxes_labels': tensor([[  0.0000,   8.0000, 533.3616, 154.0364, 640.0000, 366.4268],
        [  0.0000,   0.0000, 423.0993,  10.3340, 640.0000, 297.7388],
        [  0.0000,  10.0000, 214.7892, 606.6859, 259.2838, 640.0000],
        [  0.0000,  10.0000, 332.4818, 623.1608, 395.3741, 640.0000],
        [  0.0000,   4.0000, 213.3095, 603.9068, 231.2621, 621.8594],
        [  0.0000,  10.0000, 502.1480, 486.8242, 640.0000, 640.0000],
        [  1.0000,   2.0000, 458.0183, 174.3961, 499.5396, 279.8662],
        [  1.0000,   2.0000, 180.4387,  86.3337, 510.4552, 415.1964],
        [  1.0000,   6.0000, 239.8833, 153.9471, 249.2161, 162.8800],
        [  1.0000,   6.0000, 237.9126, 160.7557, 242.3694, 163.1556],
        [  1.0000,  10.0000, 271.6344, 133.9006, 360.6723, 172.3607],
        [  1.0000,   1.0000, 462.1844, 270.1506, 470.4734, 281.4001],
        [  1.0000,   4.0000, 412.2398, 325.6965, 508.9156, 415.8630],
        [  1.0000,   8.0000, 402.7419,  99.6409, 528.19

In [10]:
b['data_samples']['texts'][0], b['data_samples']['audio'][0]

(['traffic light',
  'suv',
  'boots',
  'person',
  'globe',
  'bus',
  'banana',
  'canned',
  'dog',
  'chainsaw',
  'board eraser',
  'airplane'],
 [tensor([0.0013, 0.0006, 0.0002,  ..., 0.0774, 0.0419, 0.0131]),
  tensor([0.0006, 0.0004, 0.0004,  ..., 0.0226, 0.0239, 0.0262]),
  tensor([ 1.0071e-03,  9.1553e-04,  1.0071e-03,  ...,  3.0518e-04,
          -9.1553e-05,  5.1880e-04]),
  tensor([ 0.0568,  0.0547,  0.0558,  ...,  0.0490,  0.0060, -0.0125]),
  tensor([ 0.0686,  0.0610,  0.0518,  ..., -0.0037, -0.0042, -0.0040]),
  tensor([ 0.0014,  0.0014,  0.0014,  ..., -0.0003, -0.0003, -0.0002]),
  tensor([-0.0425, -0.0424, -0.0395,  ..., -0.0098, -0.0155, -0.0216]),
  tensor([-0.0063,  0.0054,  0.0099,  ..., -0.0037, -0.0021,  0.0043]),
  tensor([-1.8311e-04, -3.0518e-05, -6.1035e-05,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]),
  tensor([ 0.0863,  0.0823,  0.0760,  ...,  0.0007, -0.0516,  0.0471]),
  tensor([ 1.6785e-03, -3.9673e-04, -3.2959e-03,  ..., -1.8311e-04,
     

In [9]:
import IPython
import json
import random

In [10]:
for i, a in enumerate(b['data_samples']['audio'][0]):
    print(b['data_samples']['texts'][0][i])
    IPython.display.display(IPython.display.Audio(a.numpy(), rate=16000))

airplane


sausage


swan


swan


hat


...

dumbbell


sports car


ring


person


wild bird


In [25]:
for i in train_loader.dataset.datasets[0].class_audios:
    for ii in i:
        ii[0] = '../../' + ii[0]
    # print(i)
    # break

In [26]:
train_loader.dataset.datasets[0].class_audios[:2]

[[['../../demo/obj365v1_audio/0-person.wav', 0, 28672]],
 [['../../demo/obj365v1_audio/1-sneakers.wav', 0, 13824]]]

In [40]:
len(b['data_samples']['audio'][0])

80

In [16]:
with open('../../data/audios/obj365v1_class_audio2.json', 'r') as f:
    class_audios = json.load(f)

In [25]:
for idx, audio_caps in enumerate(class_audios):
    this_wav = []
    print(audio_caps)
    if isinstance(audio_caps[0], list):
        print(audio_caps[0])
    break
    # audios

[['demo/obj365v1_audio_v2/0-0-person-3.wav', 0, 12288], ['demo/obj365v1_audio_v2/0-0-person-2.wav', 0, 9216], ['demo/obj365v1_audio_v2/0-0-person-1.wav', 0, 8192], ['demo/obj365v1_audio_v2/0-0-person-0.wav', 0, 10240]]
['demo/obj365v1_audio_v2/0-0-person-3.wav', 0, 12288]


In [21]:
class_audios_c = [[c] for c in class_audios]

In [28]:
with open('../../data/audios/obj365v1_class_audio22.json', 'w') as f:
    json.dump(class_audios_c, f)

In [26]:
for idx, audio_caps in enumerate(class_audios_c):
    this_wav = []
    # print(audio_caps)
    if isinstance(audio_caps[0], list):
        # print(audio_caps[0])
        for cls_caps in audio_caps:
            # print()
            if isinstance(cls_caps[0], list):
                cap_id = random.randrange(len(cls_caps))
                print(cap_id)
    break

1


In [27]:
len(cls_caps)

4

### mmdetection

In [24]:
import json
import numpy as np
import jsonlines

In [30]:
with open('../../data/audios/obj365v1_class_audio22.json', 'r') as f:
    obj365v1_audios = json.load(f)
    
label2audio = {}
for i, ias in enumerate(obj365v1_audios):
    for ii in ias[0]:
        ii[0] = ii[0].replace('demo/obj365v1_audio_v2/', '')
        
    label2audio[str(i)] = ias[0]

In [32]:
with open('/work/2023/yangwenhao/project/mmdetection/data/objects365v1/o365v1_audio_label_map.json', 'w') as f:
    json.dump(label2audio, f)

In [31]:
label2audio

{'0': [['0-0-person-3.wav', 0, 12288],
  ['0-0-person-2.wav', 0, 9216],
  ['0-0-person-1.wav', 0, 8192],
  ['0-0-person-0.wav', 0, 10240]],
 '1': [['1-0-sneakers-3.wav', 0, 11776],
  ['1-0-sneakers-2.wav', 0, 13312],
  ['1-0-sneakers-1.wav', 0, 12288],
  ['1-0-sneakers-0.wav', 0, 12800]],
 '2': [['2-0-chair-3.wav', 0, 7680],
  ['2-0-chair-2.wav', 0, 6656],
  ['2-0-chair-1.wav', 0, 6144],
  ['2-0-chair-0.wav', 0, 7168]],
 '3': [['3-0-hat-3.wav', 0, 6144],
  ['3-0-hat-2.wav', 0, 7680],
  ['3-0-hat-1.wav', 0, 9216],
  ['3-0-hat-0.wav', 0, 5632]],
 '4': [['4-0-lamp-3.wav', 0, 9728],
  ['4-0-lamp-2.wav', 0, 9216],
  ['4-0-lamp-1.wav', 0, 6656],
  ['4-0-lamp-0.wav', 0, 9216]],
 '5': [['5-0-bottle-3.wav', 0, 9728],
  ['5-0-bottle-2.wav', 0, 8192],
  ['5-0-bottle-1.wav', 0, 7680],
  ['5-0-bottle-0.wav', 0, 7680]],
 '6': [['6-1-shelf-3.wav', 0, 7168],
  ['6-1-shelf-2.wav', 0, 5632],
  ['6-1-shelf-1.wav', 0, 5632],
  ['6-1-shelf-0.wav', 0, 6656],
  ['6-0-cabinet-3.wav', 0, 9216],
  ['6-0-cabinet

In [14]:
label2audio[0]

[['demo/obj365v1_audio_v2/0-0-person-3.wav', 0, 12288],
 ['demo/obj365v1_audio_v2/0-0-person-2.wav', 0, 9216],
 ['demo/obj365v1_audio_v2/0-0-person-1.wav', 0, 8192],
 ['demo/obj365v1_audio_v2/0-0-person-0.wav', 0, 10240]]

In [20]:
with open('/work/2023/yangwenhao/project/mmdetection/data/objects365v1/o365v1_train_odvg.json', 'r') as f:
    data_list = [json.loads(line) for line in f.readlines()[:1024]]

In [21]:
for img in data_list:
    for inst in img['detection']['instances']:
        label = inst['label']
        audio = label2audio[label]
        if len(audio) > 1:
            rand_idx = np.random.randint(0, len(audio))
            inst['audio'] = audio[rand_idx]
        else:
            inst['audio'] = audio[0]

In [19]:
data_list

[{'filename': 'obj365_train_000000367456.jpg',
  'height': 512,
  'width': 768,
  'detection': {'instances': [{'bbox': [562.1600342016,
      192.9611816448,
      655.3520507904,
      424.1589965824],
     'label': 0,
     'category': 'person',
     'audio': ['demo/obj365v1_audio_v2/0-0-person-3.wav', 0, 12288]},
    {'bbox': [469.36535646720006,
      185.5855102464,
      549.890502912,
      431.0844115968],
     'label': 0,
     'category': 'person',
     'audio': ['demo/obj365v1_audio_v2/0-0-person-1.wav', 0, 8192]},
    {'bbox': [635.953857408, 360.0740356608, 655.0994873088, 397.344116224],
     'label': 1,
     'category': 'sneakers',
     'audio': ['demo/obj365v1_audio_v2/1-0-sneakers-3.wav', 0, 11776]},
    {'bbox': [584.6436767232, 403.470703104, 607.8736571904, 424.4031982592],
     'label': 1,
     'category': 'sneakers',
     'audio': ['demo/obj365v1_audio_v2/1-0-sneakers-2.wav', 0, 13312]},
    {'bbox': [530.27026368, 350.345275904, 548.3947753728, 387.1047973888],
   

In [23]:
with jsonlines.open('/work/2023/yangwenhao/project/mmdetection/data/objects365v1/o365v1_train_odvg_audio_1k.json', mode='w') as writer:
    writer.write_all(data_list)